# 01 — Data Exploration & Profiling
Objective: Load the raw CEMS datasets, inspect their structure, and systematically discover all data quality issues before any cleaning.



# 1. Setup & Load

In [1]:
# import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os

In [2]:
#loading the data
df = pd.read_csv('C:\\Users\\Hp\\Desktop\\Pollution_Casestudy\\data\\raw_cems_data.csv')
sensors = pd.read_csv('C:\\Users\\Hp\\Desktop\\Pollution_Casestudy\\data\\sensor_master.csv')
manual = pd.read_csv("C:\\Users\\Hp\\Desktop\\Pollution_Casestudy\\data\\manual_entries.csv")
threshold = pd.read_csv("C:\\Users\\Hp\\Desktop\\Pollution_Casestudy\\data\\regulatory_thresholds.csv")
maint = pd.read_csv("C:\\Users\\Hp\\Desktop\\Pollution_Casestudy\\data\\maintenance_logs.csv")

In [3]:
# checking the shape of datasets
print(f'raw_cems_data   : {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'sensor_master   : {sensors.shape[0]:,} rows x {sensors.shape[1]} cols')
print(f'maintenance_logs: {maint.shape[0]:,} rows x {maint.shape[1]} cols')
print(f'manual_entries  : {manual.shape[0]:,} rows x {manual.shape[1]} cols')
print(f'reg_thresholds  : {threshold.shape[0]:,} rows x {threshold.shape[1]} cols')

raw_cems_data   : 27,121 rows x 11 cols
sensor_master   : 10 rows x 9 cols
maintenance_logs: 30 rows x 5 cols
manual_entries  : 50 rows x 5 cols
reg_thresholds  : 6 rows x 3 cols


# 2.Explore the data

Inspect the raw CEMS data first since it's the primary dataset.

In [4]:
# Quick overview of the main dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27121 entries, 0 to 27120
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Record_ID        27121 non-null  str    
 1   Plant_ID         27121 non-null  str    
 2   Stack_ID         27121 non-null  str    
 3   Flow_Rate_m3_hr  21682 non-null  float64
 4   TS               27121 non-null  str    
 5   PM2.5            26726 non-null  str    
 6   SO2              26705 non-null  str    
 7   NOx              26701 non-null  str    
 8   Unit             27121 non-null  str    
 9   Status           26999 non-null  str    
 10  Lat_Lon          27121 non-null  str    
dtypes: float64(1), str(10)
memory usage: 2.3 MB


In [5]:
# First 10 rows to visually scan for obvious issues
df.head(10)

,Record_ID,Plant_ID,Stack_ID,Flow_Rate_m3_hr,TS,PM2.5,SO2,NOx,Unit,Status,Lat_Lon
0,E00001,PL-01,S-01,2033.5,01-01-2025 00:00,170.3,119.3,103.4,ug/m3,down,"13.083,80.2702"
1,E00002,PL-01,S-02,3377.9,01-01-2025 00:00,<2.0,94.2,31.6,µg/m³,MAINT,"13.0839,80.2721"
2,E00003,PL-02,S-01,3798.4,01-01-2025 00:00,123.9,NaN,101.3,mg/Nm3,FAULT,"19.0763,72.8777"
3,E00004,PL-03,S-01,3121.2,01-01-2025 00:00,117.8,147.1,100.8,ug/m3,OK,"12.9719,77.5955"
4,E00005,PL-03,S-02,3700.5,31-12-2024 24:00,37.0,15.0,157.1,mg/Nm3,FAULT,"12.9722,77.5941"
5,E00006,PL-03,S-03,1117.8,01-01-2025 00:00,197.4,48.6,111.6,ug/m3,FAULT,"12.9726,77.5939"
6,E00007,LOC-DEL-01,A-01,0.0,01-01-2025 00:00,111.9,207.0,132.8,ug/m3,DOWN,"28.6138,77.2087"
7,E00008,LOC-DEL-02,A-02,NaN,01-01-2025 00:00,187.8,159.2,147.2,ug/m3,OK,"28.6209,77.2149"
8,E00009,LOC-MUM-01,A-01,0.0,01-01-2025 00:00,196.5,80.0,124.5,ug/m3,DOWN,"19.0757,72.877"
9,E00010,LOC-BLR-01,A-01,NaN,01-01-2025 00:00,128.8,198.3,125.9,ug/m3,OK,"12.9717,77.5954"


In [6]:
# Tail too — check for end-of-file weirdness
df.tail(10)

,Record_ID,Plant_ID,Stack_ID,Flow_Rate_m3_hr,TS,PM2.5,SO2,NOx,Unit,Status,Lat_Lon
27111,E28790,LOC-BLR-01,A-01,0.0,30-01-2025 23:30,60.2,170.0,216.4,ug/m3,MAINT,"12.9726,77.5943"
27112,E28791,PL-01,S-01,4950.8,30-01-2025 23:45,211.2,BDL,-4.2,mg/Nm3,MAINT,"13.0824,80.2701"
27113,E28792,PL-01,S-02,1960.1,30-01-2025 23:45,29.2,-10.8,45.2,ug/m3,OK,"13.0836,80.2713"
27114,E28793,PL-02,S-01,2308.8,30-01-2025 23:45,209.5,98.4,138.4,ug/m3,OK,"19.0769,72.8778"
27115,E28794,PL-03,S-01,2446.9,30-01-2025 23:45,233.2,252.5,165.5,ug/m3,OK,"12.9712,77.5941"
27116,E28795,PL-03,S-02,1341.2,30-01-2025 23:45,168.3,107.7,148.2,µg/m³,FAULT,"12.9713,77.594"
27117,E28796,PL-03,S-03,2725.5,30-01-2025 23:45,34.1,182.4,NaN,ug/m3,MAINT,"12.9709,77.5956"
27118,E28797,LOC-DEL-01,A-01,0.0,30-01-2025 23:45,-9.8,139.6,104.9,ug/m3,FAULT,"28.614,77.2084"
27119,E28798,LOC-DEL-02,A-02,0.0,30-01-2025 23:45,157.9,184.4,20.0,ug/m3,error,"28.6209,77.2156"
27120,E28800,LOC-BLR-01,A-01,0.0,30-01-2025 23:45,95.2,206.7,121.9,ug/m3,FAULT,"12.9713,77.5947"


In [7]:
# Basic statistics for numeric-looking columns
df.describe()

,Flow_Rate_m3_hr
count,21682.000000
mean,2255.805101
std,1639.066173
min,0.000000
25%,0.000000
50%,2352.750000
75%,3668.475000
max,4999.800000


In [8]:
# Note: PM2.5, SO2, NOx are 'object' dtype if they contain BDL strings!
df.describe(include='all')

,Record_ID,Plant_ID,Stack_ID,Flow_Rate_m3_hr,TS,PM2.5,SO2,NOx,Unit,Status,Lat_Lon
count,27121,27121,27121,21682.000000,27121,26726,26705,26701,27121,26999,27121
unique,27121,7,5,NaN,5621,2704,2698,2711,3,17,2527
top,E00001,PL-03,A-01,NaN,31-12-2024 24:00,15.0,15.0,15.0,ug/m3,FAULT,12.9716;77.5946
freq,1,8118,8132,NaN,12,715,640,662,21753,6308,46
mean,NaN,NaN,NaN,2255.805101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,1639.066173,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,2352.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,3668.475000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 3. Missing Values Analysis

In [9]:
# Count nulls per column
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)

null_report = pd.DataFrame({
    'Null_Count': null_counts,
    'Null_%': null_pct
})

print('=== Null Values Report ===')
print(null_report)
print(f'\nTotal rows: {len(df):,}')

=== Null Values Report ===
                 Null_Count  Null_%
Record_ID                 0    0.00
Plant_ID                  0    0.00
Stack_ID                  0    0.00
Flow_Rate_m3_hr        5439   20.05
TS                        0    0.00
PM2.5                   395    1.46
SO2                     416    1.53
NOx                     420    1.55
Unit                      0    0.00
Status                  122    0.45
Lat_Lon                   0    0.00

Total rows: 27,121


# 4. checking each column

 Timestamp (TS) Analysis

In [10]:
# Check samples
print('Sample TS values:')
print(df['TS'].head(20).tolist())

Sample TS values:
['01-01-2025 00:00', '01-01-2025 00:00', '01-01-2025 00:00', '01-01-2025 00:00', '31-12-2024 24:00', '01-01-2025 00:00', '01-01-2025 00:00', '01-01-2025 00:00', '01-01-2025 00:00', '01-01-2025 00:00', '01-01-2025 00:15', '01-01-2025 00:15', '01-01-2025 00:15', '01-01-2025 00:15', '01-01-2025 00:15', '01-01-2025 00:15', '01-01-2025 00:15', '01-01-2025 00:15', '01-01-2025 00:30', '31-12-2024 19:00 UTC']


In [11]:
# Detect timestamps with hour=24 (midnight rollover issue)
ts_24 = df['TS'].str.contains(r'\b24:\d{2}', na=False, regex=True)
count_24 = ts_24.sum()
pct_24 = (count_24 / len(df) * 100).round(2)

print(f'Timestamps with hour=24: {count_24:,} ({pct_24}%)')
print('\nExamples:')
print(df.loc[ts_24, 'TS'].head(10).tolist())

Timestamps with hour=24: 797 (2.94%)

Examples:
['31-12-2024 24:00', '31-12-2024 24:15', '31-12-2024 24:45', '31-12-2024 24:45', '31-12-2024 24:45', '31-12-2024 24:00', '31-12-2024 24:00', '31-12-2024 24:30', '31-12-2024 24:30', '31-12-2024 24:00']


In [12]:
# Detect timestamps with UTC suffix
ts_utc = df['TS'].str.contains('UTC', na=False)
count_utc = ts_utc.sum()
pct_utc = (count_utc / len(df) * 100).round(2)

print(f'Timestamps with UTC suffix: {count_utc:,} ({pct_utc}%)')
print('\nExamples:')
print(df.loc[ts_utc, 'TS'].head(10).tolist())

Timestamps with UTC suffix: 1,292 (4.76%)

Examples:
['31-12-2024 19:00 UTC', '31-12-2024 19:00 UTC', '31-12-2024 19:15 UTC', '31-12-2024 19:45 UTC', '31-12-2024 20:15 UTC', '31-12-2024 20:45 UTC', '31-12-2024 21:00 UTC', '31-12-2024 21:00 UTC', '31-12-2024 21:00 UTC', '31-12-2024 21:15 UTC']


In [13]:
# Try to parse the "normal" timestamps and see what fails
ts_normal = ~ts_24 & ~ts_utc
pct_normal = (ts_normal.sum() / len(df) * 100).round(2)

print(f'Normal timestamps (parseable as-is): {ts_normal.sum():,} ({pct_normal}%)')
print(f'\n--- SUMMARY ---')
print(f'  Normal:    {ts_normal.sum():>6,} ({pct_normal}%)')
print(f'  Hour=24:   {count_24:>6,} ({pct_24}%)')
print(f'  UTC:       {count_utc:>6,} ({pct_utc}%)')
print(f'  Total:     {len(df):>6,}')
print(f'\n>> ACTION NEEDED: Fix {count_24} midnight rollovers (Rule 1), convert {count_utc} UTC timestamps to IST (Rule 17)')

Normal timestamps (parseable as-is): 25,032 (92.3%)

--- SUMMARY ---
  Normal:    25,032 (92.3%)
  Hour=24:      797 (2.94%)
  UTC:        1,292 (4.76%)
  Total:     27,121

>> ACTION NEEDED: Fix 797 midnight rollovers (Rule 1), convert 1292 UTC timestamps to IST (Rule 17)


# 5. Pollutant Columns (PM2.5, SO2, NOx)


In [14]:
# Check dtypes — if 'object', there are strings mixed in
for col in ['PM2.5', 'SO2', 'NOx']:
    print(f'{col}: dtype={df[col].dtype}')
print()
print('If dtype is "object" (not float64), there are non-numeric values mixed in.')

PM2.5: dtype=str
SO2: dtype=str
NOx: dtype=str

If dtype is "object" (not float64), there are non-numeric values mixed in.


In [15]:
# For each pollutant, classify values into: numeric, BDL string, negative, spike
def classify_pollutant(series, name):
    """Classify each value in a pollutant column."""
    results = {
        'total': len(series),
        'null': series.isnull().sum(),
        'bdl_strings': 0,
        'negatives': 0,
        'spikes': 0,
        'normal': 0,
        'bdl_examples': [],
        'neg_examples': [],
        'spike_examples': []
    }
    
    for val in series.dropna():
        val_str = str(val).strip()
        
        # Check for BDL strings like '<2.0', 'BDL', '< 5.0'
        if val_str.upper() == 'BDL' or val_str.startswith('<') or val_str.startswith('< '):
            results['bdl_strings'] += 1
            if len(results['bdl_examples']) < 5:
                results['bdl_examples'].append(val_str)
            continue
        
        # Try to parse as float
        try:
            num = float(val_str)
            if num < 0:
                results['negatives'] += 1
                if len(results['neg_examples']) < 5:
                    results['neg_examples'].append(num)
            elif num > 5000:  # suspiciously high
                results['spikes'] += 1
                if len(results['spike_examples']) < 5:
                    results['spike_examples'].append(num)
            else:
                results['normal'] += 1
        except ValueError:
            results['bdl_strings'] += 1  # catch other non-numeric
            if len(results['bdl_examples']) < 5:
                results['bdl_examples'].append(val_str)
    
    return results

for col in ['PM2.5', 'SO2', 'NOx']:
    r = classify_pollutant(df[col], col)
    print(f'\n=== {col} ===')
    print(f'  Normal values:  {r["normal"]:>6,} ({r["normal"]/r["total"]*100:.1f}%)')
    print(f'  Negatives:      {r["negatives"]:>6,} ({r["negatives"]/r["total"]*100:.1f}%)  examples: {r["neg_examples"]}')
    print(f'  BDL strings:    {r["bdl_strings"]:>6,} ({r["bdl_strings"]/r["total"]*100:.1f}%)  examples: {r["bdl_examples"]}')
    print(f'  Spikes (>5000): {r["spikes"]:>6,} ({r["spikes"]/r["total"]*100:.1f}%)  examples: {r["spike_examples"]}')
    print(f'  Null/missing:   {r["null"]:>6,} ({r["null"]/r["total"]*100:.1f}%)')
    
    
    
    
    
print('>> ACTION NEEDED:')
print('   - Remove negative values (Rule 5)')
print('   - Replace BDL strings with LOD/2 from sensor_master (Rule 11)')
print('   - Cap/filter spikes using Hampel filter (Rule 8)')


=== PM2.5 ===
  Normal values:  23,142 (85.3%)
  Negatives:       1,388 (5.1%)  examples: [-1.9, -9.4, -6.4, -4.9, -8.6]
  BDL strings:     1,391 (5.1%)  examples: ['<2.0', '< 5.0', '<2.0', '< 5.0', 'BDL']
  Spikes (>5000):    805 (3.0%)  examples: [7777.0, 7777.0, 9999.0, 7777.0, 7777.0]
  Null/missing:      395 (1.5%)

=== SO2 ===
  Normal values:  23,063 (85.0%)
  Negatives:       1,380 (5.1%)  examples: [-5.5, -11.4, -10.3, -7.8, -11.5]
  BDL strings:     1,386 (5.1%)  examples: ['< 4.0', 'BDL', '< 5.0', '<4.0', '<5.0']
  Spikes (>5000):    876 (3.2%)  examples: [9999.0, 9999.0, 8500.0, 8500.0, 8500.0]
  Null/missing:      416 (1.5%)

=== NOx ===
  Normal values:  23,087 (85.1%)
  Negatives:       1,386 (5.1%)  examples: [-8.9, -5.6, -4.7, -3.5, -6.0]
  BDL strings:     1,393 (5.1%)  examples: ['< 5.0', 'BDL', '<2.0', 'BDL', '< 5.0']
  Spikes (>5000):    835 (3.1%)  examples: [6500.0, 6500.0, 9999.0, 9999.0, 6500.0]
  Null/missing:      420 (1.5%)
>> ACTION NEEDED:
   - Remove neg

# 6. Unit Column Audit

In [16]:
# What unit values exist?
unit_counts = df['Unit'].value_counts()
print('=== Unit Value Counts ===')
print(unit_counts)
print(f'\nTotal unique values: {df["Unit"].nunique()}')

# Check for unicode characters
for val in df['Unit'].unique():
    has_unicode = any(ord(c) > 127 for c in str(val))
    if has_unicode:
        print(f'\nUnicode detected: "{val}" — contains non-ASCII characters')
        

print(f'\n>> ACTION NEEDED:')
print(f'   - Normalize unicode units to "ug/m3" (Rule 2)')
print(f'   - Convert mg/Nm3 rows: multiply pollutant values * 1000, then fix unit (Rule 14)')

=== Unit Value Counts ===
Unit
ug/m3     21753
mg/Nm3     2745
µg/m³      2623
Name: count, dtype: int64

Total unique values: 3

Unicode detected: "µg/m³" — contains non-ASCII characters

>> ACTION NEEDED:
   - Normalize unicode units to "ug/m3" (Rule 2)
   - Convert mg/Nm3 rows: multiply pollutant values * 1000, then fix unit (Rule 14)


# 7. Status Column Audit

In [17]:
# What status values exist?
status_counts = df['Status'].value_counts()
print('=== Status Value Counts ===')
print(status_counts)
print(f'\nTotal unique values: {df["Status"].nunique()}')

# Detect whitespace issues
for val in df['Status'].unique():
    stripped = str(val).strip()
    if str(val) != stripped:
        print(f'\nWhitespace issue: "{val}" → should be "{stripped}"')

# Identify canonical vs non-standard
canonical = {'OK', 'FAULT', 'MAINT'}
non_standard = [v for v in df['Status'].unique() if str(v).strip().upper() not in canonical]
print(f'\nNon-standard status values: {non_standard}')

print(f'\n>> ACTION NEEDED:')
print(f'   - Trim whitespace and normalize casing (Rule 3)')
print(f'   - Map non-standard values to OK/FAULT/MAINT (Rule 12)')

=== Status Value Counts ===
Status
FAULT          6308
OK             6276
MAINT          6239
error           630
  MAINT         616
offline         606
 maint          601
 fault          595
DOWN            593
Error 404       585
  OK            577
ok              575
maintenance     574
 ok             560
OFFLINE         556
down            554
 Ok             554
Name: count, dtype: int64

Total unique values: 17

Whitespace issue: "DOWN  " → should be "DOWN"

Whitespace issue: "  MAINT " → should be "MAINT"

Whitespace issue: "  OK  " → should be "OK"

Whitespace issue: " ok " → should be "ok"

Whitespace issue: " Ok" → should be "Ok"

Whitespace issue: "ok " → should be "ok"

Whitespace issue: " maint" → should be "maint"

Whitespace issue: " fault " → should be "fault"

Non-standard status values: ['down', 'DOWN  ', 'offline', 'maintenance', 'error', 'OFFLINE', 'Error 404', nan]

>> ACTION NEEDED:
   - Trim whitespace and normalize casing (Rule 3)
   - Map non-standard valu

# 8. Lat/Lon Validation

In [18]:
def check_latlon(df):
    
    null_count = 0
    semicolon_count = 0
    invalid_format = 0
    non_numeric = 0
    lat_out = 0
    lon_out = 0
    valid = 0
    
    for i in df.index:
        
        val = df.at[i, 'Lat_Lon']
        
        if pd.isna(val):
            null_count += 1
            continue
        
        s = str(val).strip()
        
        # check separator
        if ',' in s:
            parts = s.split(',')
        elif ';' in s:
            semicolon_count += 1
            continue
        else:
            invalid_format += 1
            continue
        
        if len(parts) != 2:
            invalid_format += 1
            continue
        
        try:
            lat = float(parts[0])
            lon = float(parts[1])
        except:
            non_numeric += 1
            continue
        
        # range check
        if lat < 6 or lat > 36:
            lat_out += 1
        elif lon < 68 or lon > 98:
            lon_out += 1
        else:
            valid += 1
        print("\n── Lat_Lon Exploration ──")
        print("Null values:", null_count)
        print("Semicolon format:", semicolon_count)
        print("Invalid format:", invalid_format)
        print("Non-numeric:", non_numeric)
        print("Latitude out of range:", lat_out)
        print("Longitude out of range:", lon_out)
        print("Valid coordinates:", valid)
    
    # 🔥 PRINT RESULTS
    check_latlon(df)
    

# 9. Gap Detection (Record_ID sequence)

In [19]:
# Extract numeric part from Record_ID (E00001 → 1)
record_nums = df['Record_ID'].str.extract(r'E(\d+)')[0].astype(int)

# Check for gaps in sequence
full_range = set(range(record_nums.min(), record_nums.max() + 1))
actual_ids = set(record_nums)
missing_ids = sorted(full_range - actual_ids)

print(f'=== Record_ID Gap Analysis ===')
print(f'Expected range: E{record_nums.min():05d} to E{record_nums.max():05d}')
print(f'Expected count: {len(full_range):,}')
print(f'Actual count:   {len(actual_ids):,}')
print(f'Missing IDs:    {len(missing_ids):,} ({len(missing_ids)/len(full_range)*100:.1f}%)')



print(f'\n>> ACTION NEEDED: Generate {len(missing_ids)} filler rows and forward-fill pollutant values (Rule 9)')

=== Record_ID Gap Analysis ===
Expected range: E00001 to E28800
Expected count: 28,800
Actual count:   27,121
Missing IDs:    1,679 (5.8%)

>> ACTION NEEDED: Generate 1679 filler rows and forward-fill pollutant values (Rule 9)


# 10. Flow Rate Analysis


In [20]:
# Check Flow_Rate_m3_hr by source type (stack vs ambient)
# Ambient sensors should have 0 or null flow rate
flow_stats = df.groupby('Stack_ID')['Flow_Rate_m3_hr'].describe()
print('=== Flow Rate by Stack_ID ===')
print(flow_stats)

# Identify ambient sensors (A-XX pattern)
ambient_mask = df['Stack_ID'].str.startswith('A-')
ambient_flow = df.loc[ambient_mask, 'Flow_Rate_m3_hr']

print(f'\nAmbient sensor flow rates:')
print(f'  Null: {ambient_flow.isnull().sum()}')
print(f'  Zero: {(ambient_flow == 0).sum()}')
print(f'  Has value: {((ambient_flow > 0) & ambient_flow.notna()).sum()}')
print(f'\n  (Ambient sensors should NOT have flow — city air is not channeled through a chimney)')

=== Flow Rate by Stack_ID ===
           count         mean          std     min       25%      50%  \
Stack_ID                                                                
A-01      4072.0     0.000000     0.000000     0.0     0.000     0.00   
A-02      1355.0     0.000000     0.000000     0.0     0.000     0.00   
S-01      8114.0  3023.466404  1150.287825  1000.8  2037.775  3022.25   
S-02      5445.0  2996.493407  1150.064022  1001.0  1994.800  3001.50   
S-03      2696.0  2990.375816  1135.004609  1000.8  2010.450  2997.50   

              75%     max  
Stack_ID                   
A-01         0.00     0.0  
A-02         0.00     0.0  
S-01      4033.05  4999.8  
S-02      3985.10  4999.2  
S-03      3957.80  4998.6  

Ambient sensor flow rates:
  Null: 5439
  Zero: 5427
  Has value: 0

  (Ambient sensors should NOT have flow — city air is not channeled through a chimney)


# 11. Reference Tables — Sensor Master

In [21]:
# Inspect sensor_master
print('=== sensor_master.csv ===')
print(sensors.to_string(index=False))
print(f'\n{sensors.shape[0]} sensors registered')

=== sensor_master.csv ===
  Plant_ID Stack_ID Source_Type Sector  Zero_Drift  Span_Mult  LOD_PM25  LOD_SO2  LOD_NOx
     PL-01     S-01       Stack Cement         0.5       1.02       2.0      4.0      5.0
     PL-01     S-02       Stack Cement         0.2       0.98       2.0      4.0      5.0
     PL-02     S-01       Stack  Steel        -0.5       1.05       2.0      5.0      5.0
     PL-03     S-01       Stack  Power         0.8       1.10       5.0      5.0      5.0
     PL-03     S-02       Stack  Power         0.1       1.00       5.0      5.0      5.0
     PL-03     S-03       Stack  Power         0.0       1.01       5.0      5.0      5.0
LOC-DEL-01     A-01     Ambient   City         0.1       1.00       1.0      2.0      2.0
LOC-DEL-02     A-02     Ambient   City         0.0       0.99       1.0      2.0      2.0
LOC-MUM-01     A-01     Ambient   City        -0.1       1.02       1.0      2.0      2.0
LOC-BLR-01     A-01     Ambient   City         0.2       1.00       1.0   

In [22]:
# Check: Does every Plant_ID + Stack_ID in raw data exist in sensor_master?
raw_combos = set(zip(df['Plant_ID'], df['Stack_ID']))
master_combos = set(zip(sensors['Plant_ID'], sensors['Stack_ID']))

orphaned = raw_combos - master_combos
unused = master_combos - raw_combos

print(f'=== Plant+Stack Mapping Validation ===')
print(f'Combos in raw_data:     {len(raw_combos)}')
print(f'Combos in sensor_master: {len(master_combos)}')
print(f'Orphaned (in raw but not master): {len(orphaned)}')
print(f'Unused (in master but not raw):   {len(unused)}')

if orphaned:
    print(f'\nOrphaned combos: {orphaned}')

print(f'\n>> ACTION NEEDED: Validate all combos match (Rule 13)')

=== Plant+Stack Mapping Validation ===
Combos in raw_data:     10
Combos in sensor_master: 10
Orphaned (in raw but not master): 0
Unused (in master but not raw):   0

>> ACTION NEEDED: Validate all combos match (Rule 13)


# 12. Maintenance Logs — Overlap Check

In [23]:
# Inspect maintenance_logs
print('=== maintenance_logs.csv ===')
print(maint.to_string(index=False))
print(f'\n{maint.shape[0]} maintenance events')

# Check for window durations
maint_copy = maint.copy()
maint_copy['Start'] = pd.to_datetime(maint_copy['Maint_Start'], format='%d-%m-%Y %H:%M')
maint_copy['End'] = pd.to_datetime(maint_copy['Maint_End'], format='%d-%m-%Y %H:%M')
maint_copy['Duration_hrs'] = (maint_copy['End'] - maint_copy['Start']).dt.total_seconds() / 3600

print(f'\nMaintenance window durations (hours):')
print(maint_copy[['Plant_ID', 'Stack_ID', 'Duration_hrs']].to_string(index=False))

print(f'\n>> ACTION NEEDED: Cross-ref with raw_cems — null pollutant values during maintenance windows (Rule 10)')

=== maintenance_logs.csv ===
  Plant_ID Stack_ID      Maint_Start        Maint_End         Technician
LOC-DEL-02     A-02 21-01-2025 23:00 22-01-2025 00:00       Suresh Patel
     PL-01     S-02 02-01-2025 00:00 02-01-2025 01:00             Team B
     PL-01     S-01 05-01-2025 11:00 05-01-2025 13:00       Suresh Patel
     PL-01     S-02 23-01-2025 13:00 23-01-2025 16:00       Vikram Singh
     PL-03     S-01 24-01-2025 07:00 24-01-2025 09:00             Team B
LOC-MUM-01     A-01 21-01-2025 23:00 22-01-2025 03:00             Team C
LOC-DEL-02     A-02 02-01-2025 19:00 02-01-2025 23:00 Maintenance Crew A
LOC-DEL-02     A-02 06-01-2025 19:00 06-01-2025 20:00       Vikram Singh
LOC-BLR-01     A-01 02-01-2025 09:00 02-01-2025 10:00             Team C
     PL-03     S-01 23-01-2025 13:00 23-01-2025 16:00         Ravi Kumar
     PL-03     S-03 23-01-2025 14:00 23-01-2025 17:00       Priya Sharma
LOC-BLR-01     A-01 22-01-2025 03:00 22-01-2025 06:00             Team C
LOC-MUM-01     A-01 27

# 13. Manual Entries — QC & PII Check

In [24]:
# Inspect manual_entries
print('=== manual_entries.csv ===')
print(manual.head(15).to_string(index=False))

=== manual_entries.csv ===
Log_ID   Plant_ID  Lab_PM25_Entry1  Lab_PM25_Entry2                                                                Inspection_Notes
 L0001      PL-03             98.7            987.0                                  Data logger memory at 45%. No action required.
 L0002      PL-01            156.4            156.7                                            Power backup tested. UPS functional.
 L0003 LOC-DEL-01             49.6             49.2                                        Routine inspection. All systems nominal.
 L0004      PL-01            123.7            124.2                                  Data logger memory at 45%. No action required.
 L0005      PL-03            132.9            132.8          Inspection passed. Signed off by Vikram (vikram.s@gov.in, 9090909090).
 L0006 LOC-BLR-01             69.9            699.0                       System broken. Call Amit at 9876543210 or amit@email.com.
 L0007      PL-01             86.8             87

In [25]:
# Double-entry QC check (Rule 16)
# Compare Lab_PM25_Entry1 vs Entry2 — flag if they differ by > 1%
e1 = manual['Lab_PM25_Entry1']
e2 = manual['Lab_PM25_Entry2']

# Percent difference
avg = (e1 + e2) / 2
pct_diff = (abs(e1 - e2) / avg * 100).round(2)

qc_fail = pct_diff > 1.0
print(f'=== Double-Entry QC Check ===')
print(f'Total entries:   {len(manual)}')
print(f'QC PASS (<=1%):  {(~qc_fail).sum()}')
print(f'QC FAIL (>1%):   {qc_fail.sum()} ({qc_fail.sum()/len(manual)*100:.0f}%)')

if qc_fail.any():
    print(f'\nFailed entries:')
    fail_df = manual[qc_fail][['Log_ID', 'Plant_ID', 'Lab_PM25_Entry1', 'Lab_PM25_Entry2']].copy()
    fail_df['Pct_Diff'] = pct_diff[qc_fail]
    print(fail_df.to_string(index=False))

print(f'\n>> ACTION NEEDED: Flag {qc_fail.sum()} entries as QC_FAIL (Rule 16)')

=== Double-Entry QC Check ===
Total entries:   50
QC PASS (<=1%):  40
QC FAIL (>1%):   10 (20%)

Failed entries:
Log_ID   Plant_ID  Lab_PM25_Entry1  Lab_PM25_Entry2  Pct_Diff
 L0001      PL-03             98.7            987.0    163.64
 L0006 LOC-BLR-01             69.9            699.0    163.64
 L0011      PL-03             17.4            174.0    163.64
 L0018 LOC-MUM-01             19.3             19.7      2.05
 L0019 LOC-BLR-01             13.3             13.7      2.96
 L0021      PL-03             17.9             17.7      1.12
 L0027 LOC-DEL-02             90.7            907.0    163.64
 L0031 LOC-DEL-02            130.0            230.2     55.64
 L0038      PL-01            172.0             23.8    151.38
 L0043      PL-03            119.5            280.5     80.50

>> ACTION NEEDED: Flag 10 entries as QC_FAIL (Rule 16)


In [26]:
# PII scan (Rule 18) — check for phone numbers and emails in notes
phone_pattern = r'\b\d{10}\b'
email_pattern = r'[\w.-]+@[\w.-]+\.\w+'

has_phone = manual['Inspection_Notes'].str.contains(phone_pattern, na=False, regex=True)
has_email = manual['Inspection_Notes'].str.contains(email_pattern, na=False, regex=True)
has_pii = has_phone | has_email

print(f'=== PII Scan in Inspection Notes ===')
print(f'Total notes:       {len(manual)}')
print(f'Contains phone #:  {has_phone.sum()}')
print(f'Contains email:    {has_email.sum()}')
print(f'Total with PII:    {has_pii.sum()} ({has_pii.sum()/len(manual)*100:.0f}%)')

if has_pii.any():
    print(f'\nSamples with PII:')
    for note in manual.loc[has_pii, 'Inspection_Notes'].head(5):
        print(f'  "{note}"')

print(f'\n>> ACTION NEEDED: Redact phone numbers and emails with [REDACTED] (Rule 18)')

=== PII Scan in Inspection Notes ===
Total notes:       50
Contains phone #:  16
Contains email:    14
Total with PII:    16 (32%)

Samples with PII:
  "Inspection passed. Signed off by Vikram (vikram.s@gov.in, 9090909090)."
  "System broken. Call Amit at 9876543210 or amit@email.com."
  "System broken. Call Amit at 9876543210 or amit@email.com."
  "Sensor drift detected. Contact Priya at 8765432109 for calibration."
  "Laser replaced by tech Suresh. Reach him at suresh.p@company.org or 7654321098."

>> ACTION NEEDED: Redact phone numbers and emails with [REDACTED] (Rule 18)


# 14. Regulatory Thresholds — Quick Look

In [27]:
# Inspect thresholds
print('=== regulatory_thresholds.csv ===')
print(threshold.to_string(index=False))
print(f'\nThese are the legal limits we will check cleaned data against (Rule 19).')
print(f'Note: Stack limits are much higher than Ambient — this is why Rule 15 (source tagging) must run BEFORE Rule 19.')

=== regulatory_thresholds.csv ===
Pollutant Source_Type  Legal_Limit_ugm3
    PM2.5     Ambient              60.0
    PM2.5       Stack             150.0
      SO2     Ambient              80.0
      SO2       Stack             200.0
      NOx     Ambient              80.0
      NOx       Stack             400.0

These are the legal limits we will check cleaned data against (Rule 19).
Note: Stack limits are much higher than Ambient — this is why Rule 15 (source tagging) must run BEFORE Rule 19.


# 15. SUMMARY OF ALL FINDINGS

Every data quality issue discovered, with the action needed.

In [28]:
# Build summary table
findings = [
    ['TS: Hour=24 timestamps', count_24, f'{pct_24}%', 'Roll to next day 00:xx', 'R1'],
    ['TS: UTC suffix', count_utc, f'{pct_utc}%', 'Convert to IST (+5:30)', 'R17'],
    ['Unit: Unicode chars', int((df['Unit'].str.contains('\u00b5|\u00b3', na=False, regex=True)).sum()), '', 'Normalize to ug/m3', 'R2'],
    ['Unit: mg/Nm3', int((df['Unit'] == 'mg/Nm3').sum()), '', 'Multiply values * 1000, fix unit', 'R14'],
    ['Status: Whitespace/messy', int(sum(1 for v in df['Status'] if str(v).strip().upper() not in canonical)), '', 'Trim + map to canonical', 'R3, R12'],
    ['Pollutants: Negatives', '~5% each', '', 'Flag and discard', 'R5'],
    ['Pollutants: BDL strings', '~5% each', '', 'Replace with LOD/2', 'R11'],
    ['Pollutants: Spikes >5000', '~3% each', '', 'Hampel filter', 'R8'],
    ['Record_ID: Gaps', len(missing_ids), f'{len(missing_ids)/len(full_range)*100:.1f}%', 'Insert filler rows + forward-fill', 'R9'],
    #['Lat_Lon: Invalid coords', len(issues), f'{len(issues)/len(df)*100:.1f}%', 'Parse and validate bounds', 'R4'],
    ['Manual: QC failures', qc_fail.sum(), f'{qc_fail.sum()/len(manual)*100:.0f}%', 'Flag as QC_FAIL', 'R16'],
    ['Manual: PII in notes', has_pii.sum(), f'{has_pii.sum()/len(manual)*100:.0f}%', 'Regex redact', 'R18'],
]

summary_df = pd.DataFrame(findings, columns=['Issue', 'Count', 'Pct', 'Action', 'Rule'])
print('=' * 90)
print('  DATA QUALITY ISSUE SUMMARY')
print('=' * 90)
print(summary_df.to_string(index=False))
print('=' * 90)
print(f'\nAdditional rules to apply (no explicit "messy" data but still needed):')
print(f'  R6:  Duplicate detection by Plant_ID + TS')
print(f'  R7:  Zero/Span calibration (sensor_master drift correction)')
print(f'  R10: Maintenance downtime cross-reference')
print(f'  R13: Plant+Stack mapping validation')
print(f'  R15: Ambient vs Stack source tagging')
print(f'  R19: Regulatory threshold exceedance flagging')
print(f'  R20: Audit trail (change log + SHA-256 integrity hash)')

  DATA QUALITY ISSUE SUMMARY
                   Issue    Count   Pct                            Action    Rule
  TS: Hour=24 timestamps      797 2.94%            Roll to next day 00:xx      R1
          TS: UTC suffix     1292 4.76%            Convert to IST (+5:30)     R17
     Unit: Unicode chars     2623                      Normalize to ug/m3      R2
            Unit: mg/Nm3     2745        Multiply values * 1000, fix unit     R14
Status: Whitespace/messy     4220                 Trim + map to canonical R3, R12
   Pollutants: Negatives ~5% each                        Flag and discard      R5
 Pollutants: BDL strings ~5% each                      Replace with LOD/2     R11
Pollutants: Spikes >5000 ~3% each                           Hampel filter      R8
         Record_ID: Gaps     1679  5.8% Insert filler rows + forward-fill      R9
     Manual: QC failures       10   20%                   Flag as QC_FAIL     R16
    Manual: PII in notes       16   32%                      Regex re

# Next Step
All issues documented. Proceed to 02_cleaning.ipynb to apply the 20 cleaning rules in dependency order.